# 4.1 — Use Document Parsing Functions

**Exam domain:** Domain 4.0 — Document Processing · **Weight:** 15%

## The problem this solves

Your accounts-payable inbox produces a few hundred PDF invoices a day. The numbers you need —
invoice number, due date, total — are visible to any human who opens the file, and invisible to
every SQL query you can write. The usual answer is a Python service outside the database that
downloads each file, runs OCR, and writes rows back. That service is a second system to secure,
monitor and pay for, and your documents now leave the warehouse to be read.

Snowflake's answer is to make a staged file something SQL can call a function on. The document
never moves, and the result is a row.

## What you will be able to do

- Turn a staged PDF into text with `AI_PARSE_DOCUMENT`, choosing between OCR and LAYOUT mode
- Process only the pages you need with `page_filter`, and read the result shape it produces
- Pull named fields straight out of a file with `AI_EXTRACT`, including table-shaped data
- Get a real error message back instead of a silent `NULL` when a file fails
- Decide when parsing first is worth an extra AI call and when it is not

## Before you start

- Run `setup/dataset.sql` once — it creates `GENAI_STUDY.PUBLIC` and the stage
  `@GENAI_STUDY.PUBLIC.DOCS_STAGE` with a directory table enabled
- Upload the files from `sample_docs/` to that stage, then run
  `ALTER STAGE GENAI_STUDY.PUBLIC.DOCS_STAGE REFRESH;`
- Your role needs the `USE AI FUNCTIONS` account privilege and one of the `SNOWFLAKE.CORTEX_USER`
  or `SNOWFLAKE.AI_FUNCTIONS_USER` database roles, plus `READ` on the stage

📖 **Snowflake documentation for this notebook**
- [AI_PARSE_DOCUMENT (SQL reference)](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
- [Parsing documents with AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)
- [AI_EXTRACT (SQL reference)](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)
- [Extracting data from documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/document-extraction)
- [TO_FILE](https://docs.snowflake.com/en/sql-reference/functions/to_file)
- [AI function privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


---
## Step one: give SQL a handle on the file

A **stage** is a named location Snowflake can read files from — either storage Snowflake manages
internally, or a bucket of yours in S3, Azure Blob or GCS that you have registered. Putting a PDF on
a stage does not make it queryable; it makes it *reachable*.

`TO_FILE` turns a stage path into a **FILE object** — a scalar value that carries the file's
location and metadata (`RELATIVE_PATH`, `STAGE`, `SIZE`, `ETAG`, `LAST_MODIFIED`, `CONTENT_TYPE`).
It does not copy the bytes into your query. Every AI function in this domain takes one of these.

```sql
TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf')
```

→ [More on TO_FILE and FILE objects](https://docs.snowflake.com/en/sql-reference/functions/to_file)

### AI_PARSE_DOCUMENT

```sql
AI_PARSE_DOCUMENT( <file_object> [, <options> ] [, <return_error_details> ] )
```

| Option key | Values | What it does |
|---|---|---|
| `mode` | `'OCR'` *(default)* | Fast, high-quality text extraction. Documented for scanned or text-heavy documents — contracts, insurance claims, manuals |
| `mode` | `'LAYOUT'` | Preserves tables, headers and layout relationships, and returns **markdown**. Recommended for most use cases, especially complex documents, and **required** for image extraction |
| `page_split` | `TRUE` / `FALSE` *(default `FALSE`)* | One result per page instead of one blob. Applies to the paged formats: PDF, PPTX, DOCX |
| `page_filter` | ARRAY of `{start, end}` | Process only these page ranges |
| `extract_images` | `TRUE` / `FALSE` *(default `FALSE`)* | LAYOUT only. Adds an `images` array |

The third argument, `return_error_details`, is a bare BOOLEAN and not part of the options object.

### What comes back

```json
{"content": "…"}                                 // default
{"pages": [{"content": "…", "index": 0}, …]}     // with page_split
```

`index` starts at **0**. Without `return_error_details`, a failure returns a plain `NULL` — no
message, no row marked bad. With it set to `TRUE` you get `{"value": …, "error": …, "metadata": …}`
instead, which is the difference between a batch you can audit and one you cannot.

→ [More on AI_PARSE_DOCUMENT options and return shapes](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)


In [ ]:
%%sql -r ocr_mode_1
-- OCR mode: fast plain-text extraction from a one-page freight invoice.
-- Note the two shapes: the raw object, and the :content path pulled out as VARCHAR.
SELECT
    AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf'),
        {'mode': 'OCR'}
    ) AS parsed_result,                                    -- {"content": "..."}
    AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf'),
        {'mode': 'OCR'}
    ):content::VARCHAR AS extracted_text;


In [ ]:
%%sql -r ocr_mode_2
-- Same call with return_error_details => TRUE.
-- You get {value, error, metadata} instead of a bare NULL when something goes wrong.
SELECT AI_PARSE_DOCUMENT(
    TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf'),
    {'mode': 'OCR'},
    TRUE
) AS parsed_with_errors;


---
## OCR or LAYOUT? The trade-off is fidelity, not price

Both modes are billed the same way — per page. So the choice is not "cheap mode versus expensive
mode". It is about what the output looks like when it lands.

- **OCR** gives you the words in reading order. On `contract_msa_harbourview.pdf`, which is mostly
  numbered prose clauses, that is all you need.
- **LAYOUT** gives you markdown: pipe tables, heading levels, column relationships. On
  `financial_statement_mgc_q3_2026.pdf`, whose meaning lives in a three-column table, OCR flattens
  the rows into a stream of numbers and LAYOUT keeps them aligned.

What LAYOUT costs you is output size. Markdown scaffolding — pipes, dashes, headings — is more
characters than the same content in plain text, and if you feed that text to another AI function
afterwards, you pay for those extra input tokens downstream. That is a second-order cost, not a
parsing cost.

`page_split` changes the shape of the answer, not its price. Use it when you want one row per page,
which is usually the case when you are about to chunk the document for search.

→ [More on choosing a parsing mode](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)


In [ ]:
%%sql -r parse_modes_1
-- LAYOUT mode on the Q3 financial statement: the tables come back as markdown,
-- so rows and columns survive the trip into text.
SELECT
    AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'financial_statement_mgc_q3_2026.pdf'),
        {'mode': 'LAYOUT'}
    ):content::VARCHAR AS layout_markdown;


In [ ]:
%%sql -r parse_modes_2
-- page_split = TRUE on a 3-page document -> {"pages":[{"content":"...","index":0}, ...]}
-- index is 0-based, so the last page of this file is index 2.
SELECT
    AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'financial_statement_mgc_q3_2026.pdf'),
        {'mode': 'LAYOUT', 'page_split': TRUE}
    ) AS pages_object;


In [ ]:
%%sql -r parse_modes_3
-- Fan the pages out into one row per page. This is the shape you want before chunking
-- and indexing a long document.
SELECT
    p.value:index::NUMBER    AS page_index,
    p.value:content::VARCHAR AS page_text
FROM TABLE(FLATTEN(
    input => AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'financial_statement_mgc_q3_2026.pdf'),
        {'mode': 'LAYOUT', 'page_split': TRUE}
    ):pages
)) p;


> ### ⚠️ Common misconceptions
>
> **"I'll cap the work with `page_limit`."**
> There is no `page_limit` option. The documented options are `mode`, `page_split`, `page_filter`
> and `extract_images`. An unknown key does not quietly cap anything — you have written an option
> the function does not recognise.
> → [AI_PARSE_DOCUMENT options](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
>
> **"`{'start': 0, 'end': 1}` gives me the first two pages."**
> It gives you exactly one page. The indexes are 0-based, `start` is inclusive and `end` is
> **exclusive**, so the first two pages of `contract_msa_harbourview.pdf` are
> `{'start': 0, 'end': 2}`. Get this wrong and you silently parse fewer pages than you think, then
> wonder why a clause from page 2 never appears.
> → [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)
>
> **"`page_filter` just narrows the input; the output looks the same."**
> Setting `page_filter` implies `page_split`, so the result is always `{"pages": [...]}`, never
> `{"content": ...}`. Reading `:content` after a `page_filter` call returns `NULL` — not an error,
> not an empty string, a `NULL` that flows into your target table and looks like a bad scan.
> → [Parsing documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)
>
> **"LAYOUT is the premium mode, so OCR must be the cheap one."**
> Parsing is billed per page and both modes cost the same per page. LAYOUT is Snowflake's
> recommendation for most documents. The only thing LAYOUT costs you is a larger text blob, which
> matters to the *next* function's input tokens, not to the parse.
> → [Parsing documents: cost considerations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)


In [ ]:
%%sql -r parse_modes_4
-- page_filter on the 3-page master services agreement.
-- Two rules that bite:
--   1. Indexes are 0-based; start is INCLUSIVE, end is EXCLUSIVE.
--      {'start':0,'end':2} returns pages 0 and 1 -- two pages, not three.
--   2. page_filter implies page_split, so the result is {"pages":[...]}, never {"content":...}.
--      Reading :content here would return NULL with no error.
SELECT
    p.value:index::NUMBER    AS page_index,
    p.value:content::VARCHAR AS page_text
FROM TABLE(FLATTEN(
    input => AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'contract_msa_harbourview.pdf'),
        {'mode': 'OCR', 'page_filter': [{'start': 0, 'end': 2}]}   -- pages 0 and 1
    ):pages
)) p;


### Pulling the pictures out too

`extract_images` is LAYOUT-only and adds an `images` array alongside the text. Each entry carries an
`id`, a bounding box (`top_left_x`, `top_left_y`, `bottom_right_x`, `bottom_right_y`) and
`image_base64`. Two documented limits shape what you get back: at most **50 images per document**,
and anything smaller than **4 × 4 pixels** is dropped — so rule lines and bullet glyphs do not
clutter the array.

There is no additional charge for this parameter. The cost you do pay is response size: base64 is
roughly a third larger than the binary it encodes, so a logo-heavy document produces a fat row.

→ [More on image extraction](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)


In [ ]:
%%sql -r parse_modes_5
-- extract_images is LAYOUT-only. The invoice carries a vendor logo, so the images
-- array should come back with at least one entry (id, bounding box, image_base64).
SELECT
    AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf'),
        {'mode': 'LAYOUT', 'extract_images': TRUE}
    ):images AS embedded_images;


---
## Doing it to every file at once

A **directory table** is the file catalogue attached to a stage. Enable it with
`DIRECTORY = (ENABLE = TRUE)` and `DIRECTORY(@stage)` becomes a queryable table of what is on the
stage: `RELATIVE_PATH`, `SIZE`, `LAST_MODIFIED`, `MD5`, `ETAG`, `FILE_URL`. That is what turns
"parse one document" into "parse the stage" with no loop.

→ [More on directory tables](https://docs.snowflake.com/en/user-guide/data-load-dirtables)

### The limits worth memorising

| Item | `AI_PARSE_DOCUMENT` |
|---|---|
| Formats | PDF, PPTX, DOCX, JPEG, JPG, PNG, TIFF, TIF, HTML, TXT |
| Max file size | 100 MB |
| Max pages per document | 2,000 |
| Max page resolution | 10,000 × 10,000 pixels (33.3 × 33.3 in at 300 DPI) |
| Privilege | `SNOWFLAKE.CORTEX_USER` database role, granted by ACCOUNTADMIN |

### How it bills

- **Paged formats** (PDF, DOCX, PPTX): each page of the document is one billed page.
- **Image formats** (JPEG, JPG, PNG, TIF, TIFF): each file is one billed page, however big it is.
- **Text formats** (HTML, TXT): each 3,000-character chunk is one billed page, including the last
  chunk. So `support_tickets.txt` bills by its length, not as "one document".

Snowflake recommends running these queries on a warehouse **no larger than MEDIUM** — larger
warehouses do not increase performance, so a bigger warehouse buys you nothing but a bigger bill.


In [ ]:
%%sql -r batch_parse
-- Parse every PDF on the stage in one set-based statement.
SELECT
    RELATIVE_PATH AS filename,
    SIZE          AS size_bytes,
    AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', RELATIVE_PATH),
        {'mode': 'LAYOUT'},
        TRUE                       -- return_error_details: one bad file cannot NULL the batch silently
    ) AS parsed
FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE)
WHERE LOWER(RELATIVE_PATH) LIKE '%.pdf'
  AND SIZE < 104857600;            -- 100 MB documented limit
-- Directory table columns: RELATIVE_PATH, SIZE, LAST_MODIFIED, MD5, ETAG, FILE_URL
-- Files that land outside Snowflake are invisible until: ALTER STAGE <name> REFRESH;


> ### 🤔 Stop and think
>
> - You parse 40,000 contracts once and store the markdown. Six months later Snowflake ships a
>   better parser. Do you re-parse the archive, or keep the text you have? What decides it — the
>   per-page bill, the risk of two versions of "the truth" in the same table, or something else?
> - `return_error_details => TRUE` makes every row bigger and every failure visible. Is there any
>   pipeline where you would deliberately prefer the silent `NULL`?
> - The 100 MB and 2,000-page limits are per document. If a customer sends you a 3,000-page
>   deposition, splitting it is a data-engineering job with its own failure modes. Who owns that
>   split — the upload process, the parsing query, or a person?


---
## AI_EXTRACT — skip the text, ask for the fields

Parsing gives you a wall of text. Most of the time what you actually wanted was four values.

`AI_EXTRACT` reads a **FILE directly**. You do not have to parse first. Running
`AI_PARSE_DOCUMENT` and then `AI_EXTRACT` on its output is two AI calls per document where one
would do — worth it only when you also want the raw text for something else, such as chunking it
into a Cortex Search service or keeping an audit copy.

### Signatures

```sql
AI_EXTRACT( <text>, <responseFormat> )
AI_EXTRACT( text => <text>, responseFormat => <fmt>, [ scores => TRUE|FALSE ] )

AI_EXTRACT( file => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE','invoice_KF-2041.pdf'), responseFormat => <fmt>,
            [ config => {'scale_factor': 1.0} ],
            [ scores => TRUE|FALSE ] )

AI_EXTRACT( model => 'db.schema.tuned_model', file => <file>, [ responseFormat => <fmt> ] )
```

### The four `responseFormat` shapes

| Shape | Example |
|---|---|
| Object of label → question | `{'total': 'What is the total amount due?'}` |
| Array of questions | `['What is the invoice number?', 'What is the due date?']` |
| Array of `[label, question]` pairs | `[['inv_no','What is the invoice number?']]` |
| JSON schema | `{'schema': {'type':'object','properties': {…}}}` — the only shape that can return **lists** and **tables** |

### What comes back

```json
{ "error": null, "response": { "total": "22384.09" } }
```

Read your fields at `:response:<label>`. With `scores => TRUE` a `scoring` block is added carrying a
0–1 confidence per field; for lists and tables you get one aggregate score, not a score per cell.

### Limits

| Item | Value |
|---|---|
| Max file size | 100 MB |
| Max pages | 125 |
| Entity questions per call | 100 |
| Table questions per call | 10 — and one table question counts as ten entity questions |
| Answer length | 512 tokens per entity question, 4,096 for a table question |
| `config => {'scale_factor': n}` | 1.0 to 4.0, default 1.0 — upscales the page before the model reads it, which helps small print |

`scale_factor` is not free: input tokens rise in proportion, and the page ceiling falls with it —
125 pages at 1.0, 62 at 2.0, 31 at 4.0. Turn it up for a 40-page scanned form and you may hit a
limit you were nowhere near before.

Formats accepted here are wider than for parsing: PDF, PNG, PPTX, PPT, EML, DOC, DOCX, JPEG, JPG,
HTM, HTML, TEXT, TXT, TIF, TIFF, BMP, GIF, WEBP, MD.

### Writing questions that work

| Do | Example |
|---|---|
| Ask a plain-English question, don't name a field | `'What is the total amount due in USD?'` beats `'amount'` |
| One value per question | Split "date and amount" into two |
| Disambiguate when the document has several of a thing | `'What is the invoice date?'` vs `'What is the payment due date?'` — `invoice_KF-2041.pdf` has both |
| State the format you want | `'…in YYYY-MM-DD format'` |
| Make optionality explicit | `'What is the purchase order number, if one is present?'` |
| Use the JSON-schema shape when the answer is a list or a table | line items, not a single total |

The model reads logos, handwriting and signatures, tables and checkmarks, so ask about them
directly rather than pre-processing the image.

→ [More on writing extraction questions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/document-extraction)


In [ ]:
%%sql -r extract_invoice_1
-- Straight from the FILE to named fields. No AI_PARSE_DOCUMENT step.
-- Expected on invoice_KF-2041.pdf: KF-2041, 2026-03-04, 2026-04-03, 22384.09, Kestrel Freight Systems.
SELECT
    AI_EXTRACT(
        file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf'),
        responseFormat => {
            'invoice_number': 'What is the invoice or reference number?',
            'invoice_date':   'What is the invoice date in YYYY-MM-DD format?',
            'due_date':       'What is the payment due date in YYYY-MM-DD format?',
            'total_amount':   'What is the total amount due in USD as a number?',
            'vendor_name':    'What company or vendor issued this invoice?'
        }
    ):response AS invoice_fields;          -- the :response wrapper is not optional


In [ ]:
%%sql -r extract_invoice_2
-- Typed columns out of the wrapper, plus the error key so a failure is visible in the row.
WITH extracted AS (
    SELECT AI_EXTRACT(
        file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_AV-8817.pdf'),
        responseFormat => {'invoice_number': 'What is the invoice reference number?',
                           'total_amount':   'What is the total due in USD?'}
    ) AS fields
)
SELECT
    fields:response:invoice_number::VARCHAR AS inv_num,
    fields:response:total_amount::FLOAT     AS total_usd,
    fields:error                            AS extract_error
FROM extracted;


In [ ]:
%%sql -r extract_invoice_3
-- Table extraction needs the JSON-schema form: a nested object whose properties are arrays,
-- plus column_ordering. invoice_TS-5530.pdf has 5 line items, so expect 5 entries per array.
-- scores => TRUE adds a scoring block; for a table it is one aggregate score, not one per cell.
SELECT AI_EXTRACT(
    file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_TS-5530.pdf'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'line_items': {
                    'description': 'The invoice line items table',
                    'type': 'object',
                    'column_ordering': ['description', 'quantity', 'amount'],
                    'properties': {
                        'description': {'description': 'Item description', 'type': 'array'},
                        'quantity':    {'description': 'Quantity',         'type': 'array'},
                        'amount':      {'description': 'Line amount',      'type': 'array'}
                    }
                }
            }
        }
    },
    scores => TRUE
) AS line_items;


> ### ⚠️ Common misconceptions
>
> **"`AI_EXTRACT` returns my fields, so `:total_amount` reads the total."**
> The return is `{"error": …, "response": {…}}`. Your fields live one level down, at
> `:response:total_amount`. Reading `:total_amount` gives a `NULL` with no error — the most common
> way to build a pipeline that runs clean and writes nothing.
> → [AI_EXTRACT return value](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)
>
> **"You have to parse a document before you can extract from it."**
> `AI_EXTRACT` accepts a FILE. Parsing first is a second AI call on the same pages, justified only
> when you want the text for its own sake. Do it reflexively across 10,000 invoices and you have
> doubled the AI spend on that pipeline for no extra field.
> → [Extracting data from documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/document-extraction)
>
> **"I'll ask for the line items with `{'items': 'List the line items'}`."**
> The plain object shape returns single values. Lists and tables require the JSON-schema shape with
> `'type': 'array'` properties. Ask for a table with an entity question and you get one truncated
> string that looks like the first row.
> → [Table extraction](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)
>
> **"`scale_factor` is a free accuracy switch."**
> It is capped at 4.0, it multiplies input tokens, and it lowers the page ceiling from 125 to 62 at
> 2.0 and 31 at 4.0. A 90-page scan that worked at 1.0 fails outright at 2.0.
> → [AI_EXTRACT config](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)


In [ ]:
%%sql -r mode_comparison
-- Does LAYOUT actually extract better than OCR on a table-heavy document?
-- Run the same question against both parses of the Q3 financial statement and compare.
WITH ocr_parse AS (
    SELECT AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'financial_statement_mgc_q3_2026.pdf'),
        {'mode': 'OCR'}
    ):content::VARCHAR AS doc_text
),
layout_parse AS (
    SELECT AI_PARSE_DOCUMENT(
        TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'financial_statement_mgc_q3_2026.pdf'),
        {'mode': 'LAYOUT'}
    ):content::VARCHAR AS doc_text
)
SELECT 'OCR' AS mode,
       AI_EXTRACT(o.doc_text,
                  {'net_income': 'What is net income for Q3 2026?'}):response:net_income::VARCHAR AS extracted_value
FROM ocr_parse o
UNION ALL
SELECT 'LAYOUT',
       AI_EXTRACT(l.doc_text,
                  {'net_income': 'What is net income for Q3 2026?'}):response:net_income::VARCHAR
FROM layout_parse l;

-- Both parses bill the same number of pages. What differs is whether the number stayed
-- attached to its row label on the way into text. Expected answer: 4,058 (USD thousands).


---
## Scenario

**Situation.** An accounts-payable team receives 200 scanned PDF invoices a day onto a stage. They
need `invoice_number`, `invoice_date`, `total_amount` and `vendor_name` in a table. Some invoices,
like `invoice_TS-5530.pdf`, carry line-item tables that finance also wants.

**Question.** Which function do you reach for, and how do you get extraction and storage into one
statement?

### Worked solution

Go straight from FILE to fields with `AI_EXTRACT`. One AI call per document instead of two. Add
`AI_PARSE_DOCUMENT` with `{'mode': 'LAYOUT'}` only if you also need the markdown — to index in
Cortex Search, or to keep the text an auditor can read.

```sql
INSERT INTO AP_INVOICES_STRUCTURED
    (filename, invoice_number, invoice_date, total_amount, vendor_name, parsed_at)
SELECT
    RELATIVE_PATH,
    fields:response:invoice_number::VARCHAR,
    TRY_TO_DATE(fields:response:invoice_date::VARCHAR),
    TRY_TO_DOUBLE(fields:response:total_amount::VARCHAR),
    fields:response:vendor_name::VARCHAR,
    CURRENT_TIMESTAMP()
FROM (
    SELECT
        RELATIVE_PATH,
        AI_EXTRACT(
            file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', RELATIVE_PATH),
            responseFormat => {
                'invoice_number': 'What is the invoice or PO reference number?',
                'invoice_date':   'What is the invoice date in YYYY-MM-DD format?',
                'total_amount':   'What is the total amount payable, as a number?',
                'vendor_name':    'What company issued this invoice?'
            }
        ) AS fields
    FROM DIRECTORY(@GENAI_STUDY.PUBLIC.DOCS_STAGE)
    WHERE LOWER(RELATIVE_PATH) LIKE '%.pdf'
      AND SIZE < 104857600
);
```

`TRY_TO_DATE` and `TRY_TO_DOUBLE` rather than casts, because the model returns strings and a single
unparseable value should not abort the insert for 199 good invoices.

Line items are a second statement using the JSON-schema table form (10 table questions per call
maximum), flattening the parallel arrays into a child table keyed by filename.

**What this design gives up:** you never keep the document text, so a question nobody thought to ask
in advance — "which invoices mention a partial shipment?" — means re-reading every PDF and paying
for it again.


---
> ### 📌 Study-guide wording versus the SQL reference
>
> Snowflake's exam study guide lists the `AI_PARSE_DOCUMENT` capabilities as
> **OCR mode · LAYOUT mode · `page_split` · `page_limit`**.
>
> The SQL reference documents `mode`, `page_split`, `page_filter` and `extract_images`.
>
> Read `page_limit` in the outline as shorthand for "restrict which pages get processed". The
> parameter that does that is `page_filter`: an array of 0-based `{start, end}` objects, `start`
> inclusive, `end` exclusive, and setting it turns on `page_split` for you. If an exam item shows
> SQL that has to run, `page_filter` is the one that runs.


---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** What is the default value of `mode`, and what does the other value return?

<details><summary>Show answer</summary>

The default is `'OCR'`. The alternative, `'LAYOUT'`, returns **markdown** and preserves tables,
headers and layout relationships. OCR is documented for fast, high-quality text extraction from
scanned or text-heavy documents; LAYOUT is recommended for most use cases, especially complex ones,
and is required if you want `extract_images`. It is tempting to assume the default is the
recommended setting — here it is not.

→ [Parsing documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

</details>

**2.** What does `AI_PARSE_DOCUMENT` return when it cannot process a file, and how do you change that?

<details><summary>Show answer</summary>

It returns `NULL` — no error, no indication which file failed. Pass `TRUE` as the third argument
(`return_error_details`) and you get `{"value": …, "error": …, "metadata": …}` instead. This is a
positional BOOLEAN argument, not a key inside the options object, which is a common place to lose
it.

→ [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)

</details>

**3.** How is parsing billed for `support_tickets.txt` compared with `invoice_KF-2041.pdf`?

<details><summary>Show answer</summary>

The one-page PDF bills as one page. The text file bills one page per **3,000-character chunk**,
including the last, partial chunk. Image formats bill one page per file regardless of dimensions.
So "pages" is a billing unit, not a property of the file — a long `.txt` can cost more than a short
PDF.

→ [Parsing documents: cost considerations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

</details>

**4.** This query returns `NULL` for every row. Why?

```sql
SELECT AI_PARSE_DOCUMENT(
    TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'contract_msa_harbourview.pdf'),
    {'mode': 'OCR', 'page_filter': [{'start': 0, 'end': 1}]}
):content::VARCHAR;
```

<details><summary>Show answer</summary>

`page_filter` implies `page_split`, so the result is `{"pages": [{"content": …, "index": 0}]}`.
There is no top-level `content` key, and reading a missing path in a VARIANT yields `NULL` rather
than an error. Read `:pages` and flatten it. The second, quieter problem: `{'start': 0, 'end': 1}`
is one page, not two, because `end` is exclusive.

→ [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document)

</details>

**5.** You want pages 2 and 3 of `financial_statement_mgc_q3_2026.pdf` (the second and third pages a
human would count). What do you write?

<details><summary>Show answer</summary>

`'page_filter': [{'start': 1, 'end': 3}]`. Indexes are 0-based, so the human's page 2 is index 1;
`start` is inclusive and `end` is exclusive, so `end` must be 3 to include index 2. The returned
objects will carry `index` 1 and 2 — the original document positions, not renumbered from zero.

→ [Parsing documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

</details>

**6.** A colleague adds `'extract_images': TRUE` to an OCR-mode call and gets no `images` key. What
is wrong?

<details><summary>Show answer</summary>

Image extraction is LAYOUT-only. In OCR mode the option does nothing. Switch `mode` to `'LAYOUT'`.
Worth knowing alongside it: at most 50 images are returned per document and anything under 4 × 4
pixels is excluded, and the parameter carries no additional charge.

→ [Parsing documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

</details>

**7.** This runs without error and writes nothing but `NULL`s. Find the bug.

```sql
SELECT AI_EXTRACT(
    file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'invoice_KF-2041.pdf'),
    responseFormat => {'total_amount': 'What is the total amount due?'}
):total_amount::FLOAT AS total;
```

<details><summary>Show answer</summary>

The accessor is missing the `response` level. `AI_EXTRACT` returns
`{"error": …, "response": {"total_amount": …}}`, so the correct path is
`:response:total_amount`. Because VARIANT path navigation returns `NULL` for a path that is not
there, nothing fails loudly — the pipeline looks healthy and the column is empty.

→ [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)

</details>

**8.** You need the 5 line items from `invoice_TS-5530.pdf` as rows. Which `responseFormat` shape,
and what limit applies?

<details><summary>Show answer</summary>

The JSON-schema shape: a nested object with `column_ordering` and properties typed as `'array'`.
The plain object-of-questions shape only returns single values. A call is limited to **10 table
questions**, and one table question is charged as ten entity questions against the 100-question
entity budget. Flatten the parallel arrays to get one row per item.

→ [AI_EXTRACT response formats](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)

</details>

**9.** A 90-page scanned document extracts fine. Someone sets `config => {'scale_factor': 2.0}` to
improve accuracy on small print, and now the call fails. Why?

<details><summary>Show answer</summary>

The maximum page count decreases with `scale_factor`: 125 pages at 1.0, 62 at 2.0, 31 at 4.0. At
2.0 a 90-page document is over the limit. Input token consumption also rises in proportion. If you
need both the scale and the page count, split the document or apply the scale only to the pages
that need it.

→ [AI_EXTRACT config](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)

</details>

**10.** You have 10,000 invoices and need four fields from each. Compare parsing first and then
extracting versus extracting from the file directly. What does each cost you?

<details><summary>Show answer</summary>

Extracting directly is one AI call per document. Parse-then-extract is two, and you pay the parse
per page whether or not the text is ever read again. Parse-then-extract buys you the document text:
without it, any future question not in your original four means re-reading all 10,000 PDFs. If your
documents are stable and your field list is not, storing the LAYOUT markdown once is usually the
cheaper bet over a year; if the field list is fixed, direct extraction wins.

→ [Extracting data from documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/document-extraction)

</details>

**11.** A team proposes an X-LARGE warehouse for a nightly parse of 50,000 pages, to "get it done
faster". What do you tell them?

<details><summary>Show answer</summary>

Snowflake recommends a warehouse no larger than MEDIUM for `AI_PARSE_DOCUMENT`, because larger
warehouses do not increase performance for these calls. The X-LARGE would bill four times the
credits per second for the same throughput. The real levers are page count — `page_filter` — and
not re-parsing files that have not changed.

→ [Parsing documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document)

</details>

**12.** You want a chat interface over `contract_msa_harbourview.pdf` and 4,000 documents like it.
Where does parsing stop and the retrieval side of the system begin?

<details><summary>Show answer</summary>

Parsing produces the text; retrieval needs that text in pieces. Parse with
`{'mode': 'LAYOUT', 'page_split': TRUE}` so you get markdown one page at a time, chunk it with
`SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(text, 'markdown', <chunk_size>, <overlap>)`, then
build a Cortex Search service over the chunk column. Snowflake recommends chunks of no more than
512 tokens, roughly 385 English words. Indexing whole 3-page documents as single rows is the usual
mistake — retrieval returns the right document and the LLM still has to find the clause.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>
